# Music Studio — Private Model Dataset Builder (v1.1)

**Accelerator: None** · **Internet: ON**

Fixes since v1.0:
- Mothersuperior tokenizer: switched to `drbaph/yue2-mothersuperior-realaudio-tokenizer-comfyui` (safetensors, pinned to the same upstream revision).
- MOSS-Music: corrected to four sharded bf16 files + index + config. **Off by default** — enable by setting `INCLUDE_MOSS = True`.

Optional speed-up: attach the already-published dataset `janpierrrewiet/music-studio-model-pack` as an input before running. The notebook will seed from `/kaggle/input/…` and only download what's missing.

Re-running **Cell 2** publishes a **new version** of the existing dataset — same URL, no need to delete anything.

In [ ]:
import hashlib, json, shutil
from pathlib import Path
from urllib.request import urlretrieve

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATASET_HANDLE = "janpierrrewiet/music-studio-model-pack"
BUILD_ROOT   = Path("/kaggle/working/music-studio-pack")
MODELS_DIR   = BUILD_ROOT / "ComfyUI" / "models"

# If True, adds ~16-18 GiB of MOSS shards. Set False to skip MOSS entirely.
INCLUDE_MOSS = False

# Optional: if the existing dataset is attached, seed from it instead of re-downloading.
SEED_FROM = Path("/kaggle/input/music-studio-model-pack")

# ---------------------------------------------------------------------------
# Model specs
# ---------------------------------------------------------------------------
MODEL_SPECS = [
    # YuE2 song generation
    ("diffusion_models", "YuE2-3B/model.safetensors",
     "m-a-p/YuE2-3B", "model.safetensors",
     None, "CC BY-NC 4.0 + M-A-P individual-creator permission"),
    ("vae", "YuE2-Vae/model.safetensors",
     "m-a-p/YuE2-Vae", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # SheetSage2 score transcription
    ("checkpoints", "SheetSage2/model.safetensors",
     "m-a-p/SheetSage2", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # --- Mothersuperior tokenizer v4 (ComfyUI safetensors conversion) ---
    ("checkpoints", "YuE2-Mothersuperior/tokenizer_head_joint_v4.safetensors",
     "drbaph/yue2-mothersuperior-realaudio-tokenizer-comfyui",
     "tokenizer_head_joint_v4.safetensors",
     None, "CC BY-NC 4.0"),
    ("checkpoints", "YuE2-Mothersuperior/nar_lora_joint_v4.safetensors",
     "drbaph/yue2-mothersuperior-realaudio-tokenizer-comfyui",
     "nar_lora_joint_v4.safetensors",
     None, "CC BY-NC 4.0"),
    ("checkpoints", "YuE2-Mothersuperior/minted_regularizer_pack.safetensors",
     "drbaph/yue2-mothersuperior-realaudio-tokenizer-comfyui",
     "minted_regularizer_pack.safetensors",
     None, "CC BY-NC 4.0"),
    ("checkpoints", "YuE2-Mothersuperior/minted_regularizer_pack.jsonl",
     "drbaph/yue2-mothersuperior-realaudio-tokenizer-comfyui",
     "minted_regularizer_pack.jsonl",
     None, "CC BY-NC 4.0"),

    # MuScriptor transcription
    ("checkpoints", "MuScriptor/medium.safetensors",
     "cocktailpeanut/muscriptor-medium", "model.safetensors",
     None, "CC BY-NC 4.0"),

    # Gemma 4 E4B QAT (text encoder)
    ("text_encoders", "gemma-4-E4B-it-qat-q4_0.gguf",
     "google/gemma-4-E4B-it-qat-q4_0-gguf", "gemma-4-E4B_q4_0-it.gguf",
     None, "Gemma Terms of Use / Prohibited Use Policy"),

    # Parakeet ASR
    ("checkpoints", "parakeet-tdt-0.6b-v3/model.safetensors",
     "nvidia/parakeet-tdt-0.6b-v3", "model.safetensors",
     None, "CC BY 4.0"),

    # Demucs 6-stem
    ("audio_separator", "htdemucs_6s.onnx",
     "StemSplitio/htdemucs-6s-onnx", "htdemucs_6s.onnx",
     None, "MIT"),
]

if INCLUDE_MOSS:
    for i in range(1, 5):
        MODEL_SPECS.append((
            "checkpoints", f"MOSS-Music-8B-Instruct/model-0000{i}-of-00004.safetensors",
            "OpenMOSS-Team/MOSS-Music-8B-Instruct",
            f"model-0000{i}-of-00004.safetensors",
            None, "Apache-2.0",
        ))
    MODEL_SPECS += [
        ("checkpoints", "MOSS-Music-8B-Instruct/model.safetensors.index.json",
         "OpenMOSS-Team/MOSS-Music-8B-Instruct", "model.safetensors.index.json",
         None, "Apache-2.0"),
        ("checkpoints", "MOSS-Music-8B-Instruct/config.json",
         "OpenMOSS-Team/MOSS-Music-8B-Instruct", "config.json",
         None, "Apache-2.0"),
    ]

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_hf_file(repo_id: str, filename: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
    print(f"  ↓ {url}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f"    saved {dest} ({dest.stat().st_size / 1024**2:.1f} MiB)")

def seed_from_previous(dest: Path) -> bool:
    """If the file exists inside the attached previous dataset, copy it over. Returns True on success."""
    if not SEED_FROM.exists():
        return False
    rel = dest.relative_to(BUILD_ROOT)
    candidate = SEED_FROM / rel
    if candidate.exists() and candidate.stat().st_size > 0:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, dest)
        print(f"  ⤵ seeded from previous dataset: {rel}")
        return True
    return False

# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------
print("=== Music Studio Model Pack Builder (v1.1) ===")
if SEED_FROM.exists():
    print(f"Seeding from attached previous dataset: {SEED_FROM}")
else:
    print("No previous dataset attached — will download everything fresh.")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
manifest_entries, failures = [], []

for subdir, rel_filename, repo_id, hf_filename, expected_sha, license_str in MODEL_SPECS:
    dest = MODELS_DIR / subdir / rel_filename

    if dest.exists():
        print(f"✓ Already present: {dest}")
    elif seed_from_previous(dest):
        pass  # seeded
    else:
        print(f"↓ Downloading {rel_filename} from {repo_id} ...")
        try:
            download_hf_file(repo_id, hf_filename, dest)
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            failures.append((rel_filename, str(e)))
            continue

    actual_sha = sha256_file(dest)
    if expected_sha and actual_sha != expected_sha:
        print(f"  ⚠ SHA256 mismatch for {rel_filename}: got {actual_sha}")
    else:
        print(f"  ✓ SHA256: {actual_sha[:16]}...")

    manifest_entries.append({
        "path": str(dest.relative_to(BUILD_ROOT)),
        "source_repo": repo_id,
        "source_file": hf_filename,
        "sha256": actual_sha,
        "size_bytes": dest.stat().st_size,
        "license": license_str,
    })

# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
manifest = {
    "name": "Music Studio Model Pack",
    "version": "1.1.0",
    "description": "Model weights for Music Studio / Stem Splitter ComfyUI canvas",
    "metadata_license": "other",
    "includes_moss": INCLUDE_MOSS,
    "components": manifest_entries,
    "failures": failures,
}
(BUILD_ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nManifest written to {BUILD_ROOT / 'manifest.json'}")

if failures:
    print(f"\n⚠ {len(failures)} model(s) failed. Re-run Cell 1 to retry; existing files are reused.")
else:
    print("\n✅ All models downloaded / seeded / verified successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Publish as a NEW VERSION of the existing private Kaggle Dataset
# ---------------------------------------------------------------------------
import kagglehub

print("=== Publishing new version of private dataset ===")
print(f"Handle:    {DATASET_HANDLE}")
print(f"Local dir: {BUILD_ROOT}")
print("Note: same URL — existing attachments in ComfyUI notebooks keep working.")

try:
    kagglehub.dataset_upload(
        DATASET_HANDLE,
        str(BUILD_ROOT),
        version_notes="v1.1 — fixed Mothersuperior tokenizer sources; MOSS optional",
        ignore_patterns=["*.part", ".*"],
    )
    print("\n✅ NEW VERSION SUBMITTED.")
    print("Wait for processing on the dataset page, then start the ComfyUI notebook as usual.")
except Exception as e:
    print(f"\n❌ Upload failed: {e}")
    print("Verified files remain in:", BUILD_ROOT)
    print("Enable the KAGGLE_API_TOKEN secret and re-run, or Save Version with outputs.")
    raise